# 🛡️ Passive Sonar Threat Classification — ConvViT
**Dataset:** ShipsEar / DS3500 | **FPGA Target:** Zynq UltraScale+ ZU9EG

```
WAV → Mel Spectrogram → CNN Backbone → Feature Tokens → Dynamic-Scale Transformer → 5 Classes
```

| Class | Ship Type | Defense Label |
|---|---|---|
| 0 | Motorboat | ⚠️ Potential Hostile |
| 1 | Fishing Vessel | 👁️ Monitor |
| 2 | Cargo / Tanker | ✅ Non-Threat |
| 3 | Tugboat | 👁️ Monitor |
| 4 | Environment / Noise | 🌊 Clear Water |

## ♻️ Resume System
Every expensive step cached to disk. Reopen → each phase auto-skips if cache exists.

| Cache File | Skips |
|---|---|
| `cache/spectrograms.npz` | WAV→Mel conversion |
| `cache/splits.npz` | Train/val/test split |
| `best_sonar_convvit.pt` | Full retraining |
| `cache/training_history.npz` | Training curves |
| `cache/eval_results.npz` | FP32 + INT8 evaluation |
| `cache/jamming_results.npz` | Noise sweep |

**Force rerun any phase:** delete its cache file, rerun that cell only.

## Phase 1 — Install & Imports

In [1]:
import os, time, random, warnings
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import librosa
from pathlib import Path
from tqdm import tqdm
from collections import Counter

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.quantization import quantize_dynamic
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split

warnings.filterwarnings('ignore')
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.backends.mps.is_available(): torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
print(f'Device : {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU    : Apple Silicon (MPS)')
    # MPS uses unified memory

# Cache setup
CACHE_DIR     = Path('./cache'); CACHE_DIR.mkdir(exist_ok=True)
CACHE_SPEC    = CACHE_DIR / 'spectrograms.npz'
CACHE_SPLITS  = CACHE_DIR / 'splits.npz'
CACHE_HISTORY = CACHE_DIR / 'training_history.npz'
CACHE_EVAL    = CACHE_DIR / 'eval_results.npz'
CACHE_JAMMING = CACHE_DIR / 'jamming_results.npz'
MODEL_PATH    = Path('./best_sonar_convvit.pt')

def cache_status():
    files = {'Spectrograms':CACHE_SPEC,'Splits':CACHE_SPLITS,
             'Model':MODEL_PATH,'History':CACHE_HISTORY,
             'Eval':CACHE_EVAL,'Jamming':CACHE_JAMMING}
    print('\nCache status:')
    for label, path in files.items():
        if path.exists():
            print(f'  ✅ {label:15s} → {path.name} ({path.stat().st_size/1e6:.1f} MB)')
        else:
            print(f'  🔄 {label:15s} → will compute')

cache_status()

Device : cpu

Cache status:
  🔄 Spectrograms    → will compute
  🔄 Splits          → will compute
  🔄 Model           → will compute
  🔄 History         → will compute
  🔄 Eval            → will compute
  🔄 Jamming         → will compute


## Phase 2 — File Collection

In [2]:
# ── CHANGE THIS PATH TO YOUR FOLDER ─────────────────────────────────────────
SHIPSEAR_ROOT = Path('DS3500/ShipsEar_extracted/shipsear_5s_16k')
# ─────────────────────────────────────────────────────────────────────────────

NUM_CLASSES   = 5
CLASS_NAMES   = ['Motorboat','Fishing','Cargo/Tanker','Tugboat','Environment']
DEFENSE_LABEL = ['⚠ Hostile','👁 Monitor','✅ Non-Threat','👁 Monitor','🌊 Clear']

all_samples = []
for cid in range(NUM_CLASSES):
    cdir = SHIPSEAR_ROOT / str(cid)
    if not cdir.exists():
        print(f'WARNING: {cdir} not found'); continue
    for f in sorted(cdir.rglob('*.wav')):
        all_samples.append((str(f), cid))

label_counts = Counter(lbl for _, lbl in all_samples)
print(f'Total WAV files: {len(all_samples)}')
print(f'\n{"CID":>4} {"Class":>15} {"Count":>7} {"Pct":>7}  Defense')
print('-'*55)
for cid in range(NUM_CLASSES):
    n = label_counts[cid]
    print(f'{cid:>4} {CLASS_NAMES[cid]:>15} {n:>7} {n/len(all_samples)*100:>6.1f}%  {DEFENSE_LABEL[cid]}')

Total WAV files: 2223

 CID           Class   Count     Pct  Defense
-------------------------------------------------------
   0       Motorboat     369   16.6%  ⚠ Hostile
   1         Fishing     301   13.5%  👁 Monitor
   2    Cargo/Tanker     843   37.9%  ✅ Non-Threat
   3         Tugboat     486   21.9%  👁 Monitor
   4     Environment     224   10.1%  🌊 Clear


## Phase 3 — WAV → Mel Spectrogram (Cached)
> **First run:** Converts all WAV files (~5-10 min). Saves to `cache/spectrograms.npz`.
> **Next run:** Loads in ~3 seconds. WAV files never touched again.

**Real-world fix:** Global Z-score normalization (not per-sample). Preserves amplitude differences between classes.

In [3]:
SR, CLIP_DUR = 16000, 5
N_MELS, N_FFT, HOP = 128, 512, 256
IMG_H, IMG_W = 128, 304
PATCH_SIZE   = 16
pH = IMG_H // PATCH_SIZE   # 8
pW = IMG_W // PATCH_SIZE   # 19
NUM_PATCHES = pH * pW       # 152
print(f'Spectrogram : ({IMG_H}, {IMG_W})')
print(f'Patches     : {pH} x {pW} = {NUM_PATCHES} tokens')

def wav_to_raw_melspec(path):
    """WAV -> raw log-Mel dB. No normalization (done globally)."""
    y, _ = librosa.load(path, sr=SR, duration=CLIP_DUR, mono=True)
    target = SR * CLIP_DUR
    y = np.pad(y, (0, max(0, target - len(y))))[:target]
    mel     = librosa.feature.melspectrogram(y=y, sr=SR, n_fft=N_FFT,
                                              hop_length=HOP, n_mels=N_MELS)
    log_mel = librosa.power_to_db(mel, ref=np.max)
    w = log_mel.shape[1]
    if w < IMG_W: log_mel = np.pad(log_mel, ((0,0),(0, IMG_W-w)))
    else:         log_mel = log_mel[:, :IMG_W]
    return log_mel.astype(np.float32)

if CACHE_SPEC.exists():
    print('✅ Loading spectrograms from cache...')
    d           = np.load(CACHE_SPEC)
    all_specs   = d['specs']
    all_labels  = d['labels']
    GLOBAL_MEAN = float(d['global_mean'])
    GLOBAL_STD  = float(d['global_std'])
    print(f'   {len(all_specs)} specs | mean={GLOBAL_MEAN:.2f} dB | std={GLOBAL_STD:.2f} dB')
else:
    print(f'🔄 Converting {len(all_samples)} WAV files (runs ONCE)...')
    specs, labels, failed = [], [], []
    for path, lbl in tqdm(all_samples, desc='WAV→Mel'):
        try:
            specs.append(wav_to_raw_melspec(path))
            labels.append(lbl)
        except Exception as e:
            failed.append((path, str(e)))
    all_specs   = np.stack(specs).astype(np.float32)
    all_labels  = np.array(labels, dtype=np.int64)
    GLOBAL_MEAN = float(all_specs.mean())
    GLOBAL_STD  = float(all_specs.std())
    print(f'Global stats: mean={GLOBAL_MEAN:.2f} dB, std={GLOBAL_STD:.2f} dB')
    np.savez_compressed(CACHE_SPEC, specs=all_specs, labels=all_labels,
                        global_mean=GLOBAL_MEAN, global_std=GLOBAL_STD)
    print(f'✅ Saved → {CACHE_SPEC.name} ({CACHE_SPEC.stat().st_size/1e6:.1f} MB)')
    if failed: print(f'⚠  {len(failed)} files failed')

print(f'Specs  : {all_specs.shape}')
print(f'Labels : {all_labels.shape}')

Spectrogram : (128, 304)
Patches     : 8 x 19 = 152 tokens
🔄 Converting 2223 WAV files (runs ONCE)...


WAV→Mel: 100%|██████████| 2223/2223 [00:28<00:00, 78.42it/s] 


Global stats: mean=-32.06 dB, std=10.80 dB
✅ Saved → spectrograms.npz (297.4 MB)
Specs  : (2223, 128, 304)
Labels : (2223,)


In [ ]:
# Visualize one sample per class
fig, axes = plt.subplots(1, 5, figsize=(22, 4))
fig.suptitle('Log-Mel Spectrograms — ShipsEar (from cache)', fontsize=12, fontweight='bold')
for cid, ax in enumerate(axes):
    idx      = np.where(all_labels == cid)[0][0]
    spec_vis = (all_specs[idx] - GLOBAL_MEAN) / (GLOBAL_STD + 1e-8)
    ax.imshow(spec_vis, aspect='auto', origin='lower', cmap='magma')
    ax.set_title(f'Class {cid}: {CLASS_NAMES[cid]}\n{DEFENSE_LABEL[cid]}', fontsize=10)
    ax.set_xlabel('Time frames'); ax.set_ylabel('Mel bins')
plt.tight_layout()
plt.savefig('sonar_spectrograms.png', dpi=150)
plt.show()

## Phase 4 — Splits & Dataset (Cached)
**Augmentations (training only):** time-shift, freq mask, time mask, SNR noise, amplitude scaling.

In [ ]:
if CACHE_SPLITS.exists():
    print('✅ Loading splits from cache...')
    sp = np.load(CACHE_SPLITS)
    tr_idx, vl_idx, te_idx = sp['tr_idx'], sp['vl_idx'], sp['te_idx']
else:
    print('🔄 Computing stratified splits...')
    idx = np.arange(len(all_specs))
    tr_idx, tmp = train_test_split(idx, test_size=0.30,
                                    stratify=all_labels, random_state=SEED)
    vl_idx, te_idx = train_test_split(tmp, test_size=0.50,
                                       stratify=all_labels[tmp], random_state=SEED)
    np.savez(CACHE_SPLITS, tr_idx=tr_idx, vl_idx=vl_idx, te_idx=te_idx)
    print(f'✅ Saved → {CACHE_SPLITS.name}')

print(f'Train:{len(tr_idx)} | Val:{len(vl_idx)} | Test:{len(te_idx)}')

counts  = torch.tensor([label_counts[i] for i in range(NUM_CLASSES)], dtype=torch.float)
weights = (1.0/counts / (1.0/counts).sum() * NUM_CLASSES).to(DEVICE)
print(f'Class weights: {[f"{w:.2f}" for w in weights.cpu()]}')


class SonarDataset(Dataset):
    def __init__(self, specs, labels, indices, mean=0.0, std=1.0, augment=False):
        self.specs   = specs
        self.labels  = labels
        self.indices = indices
        self.mean    = mean
        self.std     = std
        self.augment = augment

    def __len__(self): return len(self.indices)

    def __getitem__(self, i):
        idx  = self.indices[i]
        spec = self.specs[idx].copy()
        lbl  = int(self.labels[idx])

        if self.augment:
            # 1. Time shift
            if random.random() < 0.5:
                spec = np.roll(spec, random.randint(-30, 30), axis=1)
            # 2. Frequency mask
            if random.random() < 0.5:
                f0 = random.randint(0, IMG_H - 20)
                fw = random.randint(5, 20)
                spec[f0:f0+fw, :] = self.mean
            # 3. Time mask
            if random.random() < 0.5:
                t0 = random.randint(0, IMG_W - 40)
                tw = random.randint(10, 40)
                spec[:, t0:t0+tw] = self.mean
            # 4. SNR noise (10-30 dB)
            if random.random() < 0.4:
                snr  = random.uniform(10, 30)
                spec = spec + (10**(-snr/20.0)) * np.random.randn(*spec.shape).astype(np.float32) * self.std
            # 5. Amplitude scale (distance simulation)
            if random.random() < 0.3:
                spec = spec * random.uniform(0.7, 1.3)

        # Global Z-score normalization
        spec = (spec - self.mean) / (self.std + 1e-8)
        return torch.tensor(spec, dtype=torch.float32).unsqueeze(0), lbl


BATCH    = 32
train_ds = SonarDataset(all_specs, all_labels, tr_idx, GLOBAL_MEAN, GLOBAL_STD, augment=True)
val_ds   = SonarDataset(all_specs, all_labels, vl_idx, GLOBAL_MEAN, GLOBAL_STD)
test_ds  = SonarDataset(all_specs, all_labels, te_idx, GLOBAL_MEAN, GLOBAL_STD)
train_dl = DataLoader(train_ds, batch_size=BATCH, shuffle=True,  num_workers=0, pin_memory=True)
val_dl   = DataLoader(val_ds,   batch_size=BATCH, shuffle=False, num_workers=0, pin_memory=True)
test_dl  = DataLoader(test_ds,  batch_size=BATCH, shuffle=False, num_workers=0, pin_memory=True)

x, y = next(iter(train_dl))
print(f'Batch shape : {x.shape} | min={x.min():.2f} max={x.max():.2f}')

## Phase 5 — ConvViT Model Definition
**CNN Backbone** extracts local features (propeller harmonics, cavitation). Works with small data.
**Transformer** captures global temporal-frequency dependencies on top of CNN features.
**Dynamic-Scale Attention** is the core contribution — hardware-equivalent INT8 robustness.

In [ ]:
class ConvBlock(nn.Module):
    """2x Conv3x3 + BN + GELU + optional Pool. FPGA: DSP48 systolic chain."""
    def __init__(self, in_ch, out_ch, pool=True):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.GELU(),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.GELU())
        self.pool = nn.MaxPool2d(2,2) if pool else nn.Identity()
    def forward(self, x): return self.pool(self.conv(x))


class CNNBackbone(nn.Module):
    """
    Input  : (B,  1, 128, 304)
    Stage1 : (B, 32,  64, 152)  MaxPool2d
    Stage2 : (B, 64,  32,  76)  MaxPool2d
    Stage3 : (B,128,   8,  19)  AdaptiveAvgPool2d
    Output : 152 tokens of dim 128
    """
    def __init__(self):
        super().__init__()
        self.stage1 = ConvBlock(1,   32, pool=True)
        self.stage2 = ConvBlock(32,  64, pool=True)
        self.stage3 = nn.Sequential(
            nn.Conv2d(64, 128, 3, padding=1, bias=False),
            nn.BatchNorm2d(128), nn.GELU(),
            nn.Conv2d(128, 128, 3, padding=1, bias=False),
            nn.BatchNorm2d(128), nn.GELU(),
            nn.AdaptiveAvgPool2d((pH, pW)))
    def forward(self, x):
        x = self.stage1(x)
        x = self.stage2(x)
        x = self.stage3(x)
        return x


class DynamicScaleAttention(nn.Module):
    """
    CORE CONTRIBUTION:
    Static INT8 attention collapses when sonar QKt range spikes under jamming.
    Solution: per-step max-reduce -> normalize -> learned scale register.
    FPGA: max-reduce tree runs parallel to attention pipeline.
    """
    def __init__(self, embed_dim, num_heads, dropout=0.1, dynamic=True):
        super().__init__()
        self.H       = num_heads
        self.Dh      = embed_dim // num_heads
        self.dynamic = dynamic
        self.qkv     = nn.Linear(embed_dim, 3*embed_dim, bias=False)
        self.proj    = nn.Linear(embed_dim, embed_dim)
        self.drop    = nn.Dropout(dropout)
        self.alpha   = nn.Parameter(torch.ones(1))

    def forward(self, x):
        B, N, D = x.shape
        qkv = self.qkv(x).reshape(B, N, 3, self.H, self.Dh).permute(2,0,3,1,4)
        q, k, v = qkv.unbind(0)
        attn = q @ k.transpose(-2, -1)
        if self.dynamic:
            drange = attn.abs().amax(dim=(-2,-1), keepdim=True).clamp(min=1.0)
            attn   = (attn / drange) * self.alpha
        else:
            attn = attn * (self.Dh ** -0.5)
        w   = self.drop(F.softmax(attn, dim=-1))
        out = (w @ v).transpose(1,2).reshape(B, N, D)
        return self.proj(out), w


class TransformerBlock(nn.Module):
    def __init__(self, embed_dim, num_heads, mlp_ratio=4.0, dropout=0.1, dynamic=True):
        super().__init__()
        self.norm1 = nn.LayerNorm(embed_dim)
        self.attn  = DynamicScaleAttention(embed_dim, num_heads, dropout, dynamic)
        self.norm2 = nn.LayerNorm(embed_dim)
        mlp_d = int(embed_dim * mlp_ratio)
        self.mlp = nn.Sequential(
            nn.Linear(embed_dim, mlp_d), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(mlp_d, embed_dim), nn.Dropout(dropout))
    def forward(self, x):
        a, w = self.attn(self.norm1(x))
        x = x + a
        x = x + self.mlp(self.norm2(x))
        return x, w


class ConvViT(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES, embed_dim=256,
                 depth=3, num_heads=8, mlp_ratio=4.0, dropout=0.1, dynamic=True):
        super().__init__()
        self.cnn    = CNNBackbone()
        self.proj   = nn.Linear(128, embed_dim)
        N = NUM_PATCHES
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.pos_embed = nn.Parameter(torch.randn(1, N+1, embed_dim) * 0.02)
        self.pos_drop  = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([
            TransformerBlock(embed_dim, num_heads, mlp_ratio, dropout, dynamic)
            for _ in range(depth)])
        self.norm = nn.LayerNorm(embed_dim)
        self.head = nn.Sequential(
            nn.Linear(embed_dim, embed_dim//2), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(embed_dim//2, num_classes))
        self._init()

    def _init(self):
        nn.init.trunc_normal_(self.cls_token, std=0.02)
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.trunc_normal_(m.weight, std=0.02)
                if m.bias is not None: nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out')

    def forward(self, x, return_attn=False):
        B  = x.size(0)
        f  = self.cnn(x)
        f  = f.flatten(2).transpose(1, 2)
        f  = self.proj(f)
        cls = self.cls_token.expand(B, -1, -1)
        x   = torch.cat([cls, f], dim=1)
        x   = self.pos_drop(x + self.pos_embed)
        aws = []
        for blk in self.blocks:
            x, w = blk(x); aws.append(w)
        out = self.head(self.norm(x)[:, 0])
        return (out, aws) if return_attn else out


model = ConvViT(dynamic=True).to(DEVICE)
total = sum(p.numel() for p in model.parameters())
cnn_p = sum(p.numel() for p in model.cnn.parameters())
print(f'Total params     : {total:,}')
print(f'  CNN backbone   : {cnn_p:,} ({cnn_p/total*100:.1f}%)')
print(f'  Transformer    : {total-cnn_p:,} ({(total-cnn_p)/total*100:.1f}%)')
print(f'FP32 size        : {total*4/1e6:.2f} MB')
print(f'INT8 size (est.) : {total*1/1e6:.2f} MB')

# Verify forward pass
with torch.no_grad():
    dummy = torch.randn(2, 1, IMG_H, IMG_W).to(DEVICE)
    out   = model(dummy)
    print(f'Forward OK: {tuple(dummy.shape)} -> {tuple(out.shape)}')

## Phase 6 — Training (Cached)
> **First run:** Trains from scratch. **Next run:** Loads checkpoint, skips training.

**Real-world fixes:** Mixup augmentation + linear LR warmup + early stopping (patience=12).

In [ ]:
def mixup(x, y, num_classes, alpha=0.3):
    lam  = np.random.beta(alpha, alpha)
    idx  = torch.randperm(x.size(0), device=x.device)
    x_m  = lam*x + (1-lam)*x[idx]
    y_a  = F.one_hot(y, num_classes).float()
    y_b  = F.one_hot(y[idx], num_classes).float()
    return x_m, lam*y_a + (1-lam)*y_b


if MODEL_PATH.exists():
    print(f'✅ Checkpoint found -> loading {MODEL_PATH.name}')
    model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
    print('   Training SKIPPED. Delete best_sonar_convvit.pt to retrain.')
else:
    print('🔄 Training ConvViT from scratch...\n')
    EPOCHS, LR, WARMUP_EP, PATIENCE = 60, 1e-3, 5, 12
    criterion = nn.CrossEntropyLoss(weight=weights, label_smoothing=0.1)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.05)

    def lr_lambda(ep):
        if ep < WARMUP_EP: return (ep+1)/WARMUP_EP
        p = (ep-WARMUP_EP)/max(EPOCHS-WARMUP_EP, 1)
        return max(1e-3, 0.5*(1+np.cos(np.pi*p)))
    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

    history = {'tr_loss':[],'vl_loss':[],'tr_acc':[],'vl_acc':[]}
    best_acc, no_imp = 0.0, 0

    def train_epoch():
        model.train()
        tl, correct, total = 0.0, 0, 0
        for x, y in tqdm(train_dl, leave=False, desc='train'):
            x, y = x.to(DEVICE), y.to(DEVICE)
            if random.random() < 0.5:
                xm, ym = mixup(x, y, NUM_CLASSES)
                logits = model(xm)
                loss   = -(ym * F.log_softmax(logits, -1)).sum(-1).mean()
            else:
                logits = model(x)
                loss   = criterion(logits, y)
            optimizer.zero_grad(); loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            tl += loss.item()*x.size(0)
            correct += (logits.argmax(1)==y).sum().item()
            total   += x.size(0)
        return tl/total, correct/total

    def val_epoch():
        model.eval(); vl, correct, total = 0.0, 0, 0
        with torch.no_grad():
            for x, y in tqdm(val_dl, leave=False, desc='val  '):
                x, y = x.to(DEVICE), y.to(DEVICE)
                logits = model(x)
                vl    += criterion(logits, y).item()*x.size(0)
                correct += (logits.argmax(1)==y).sum().item()
                total   += x.size(0)
        return vl/total, correct/total

    for ep in range(1, EPOCHS+1):
        t0 = time.time()
        tl, ta = train_epoch()
        vl, va = val_epoch()
        scheduler.step()
        history['tr_loss'].append(tl); history['tr_acc'].append(ta)
        history['vl_loss'].append(vl); history['vl_acc'].append(va)
        tag = ''
        if va > best_acc:
            best_acc, no_imp = va, 0
            torch.save(model.state_dict(), MODEL_PATH); tag=' ✓'
        else:
            no_imp += 1
        lr_now = optimizer.param_groups[0]['lr']
        print(f'Ep{ep:02d}/{EPOCHS} TrL={tl:.3f} TrA={ta:.3f} '
              f'VlL={vl:.3f} VlA={va:.3f} lr={lr_now:.1e} '
              f'{time.time()-t0:.1f}s{tag}')
        if no_imp >= PATIENCE:
            print(f'\n⏹ Early stop at epoch {ep}'); break

    np.savez(CACHE_HISTORY, **{k:np.array(v) for k,v in history.items()})
    print(f'\n✅ Best Val Acc: {best_acc:.4f}')

In [ ]:
if CACHE_HISTORY.exists():
    h = np.load(CACHE_HISTORY)
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(14, 5))
    a1.plot(h['tr_loss'], color='royalblue', label='Train')
    a1.plot(h['vl_loss'], color='tomato',    label='Val')
    a1.set(title='Loss', xlabel='Epoch'); a1.legend(); a1.grid(alpha=0.3)
    a2.plot(h['tr_acc'], color='royalblue', label='Train')
    a2.plot(h['vl_acc'], color='tomato',    label='Val')
    a2.set(title='Accuracy', xlabel='Epoch'); a2.legend(); a2.grid(alpha=0.3)
    plt.tight_layout(); plt.savefig('training_curves.png', dpi=150); plt.show()
else:
    print('No history cache — run Phase 6 first.')

## Phase 7 — INT8 Quantization + Evaluation (Cached)

In [ ]:
model.load_state_dict(torch.load(MODEL_PATH, map_location='cpu'))
model_fp32 = model.cpu().eval()
model_int8 = quantize_dynamic(model_fp32, {nn.Linear}, dtype=torch.qint8)

def model_mb(m):
    torch.save(m.state_dict(), '_tmp.pt')
    s = os.path.getsize('_tmp.pt')/1e6; os.remove('_tmp.pt'); return s

fp32_mb = model_mb(model_fp32)
int8_mb = model_mb(model_int8)
print(f'FP32:{fp32_mb:.2f} MB | INT8:{int8_mb:.2f} MB | Compression:{fp32_mb/int8_mb:.2f}x')

test_dl_cpu = DataLoader(test_ds, batch_size=32, shuffle=False, num_workers=0)

def eval_model(m, loader, tag):
    m.eval(); preds_all, labels_all = [], []
    t0 = time.time()
    with torch.no_grad():
        for x, y in tqdm(loader, leave=False, desc=tag):
            preds_all.extend(m(x).argmax(1).numpy())
            labels_all.extend(y.numpy())
    t = time.time()-t0
    p, l = np.array(preds_all), np.array(labels_all)
    acc = (p==l).mean()
    print(f'\n── {tag} ── Acc={acc:.4f} | {t/len(l)*1000:.2f} ms/sample')
    print(classification_report(l, p, target_names=CLASS_NAMES))
    return p, l, acc

if CACHE_EVAL.exists():
    print('✅ Loading eval from cache...')
    ev = np.load(CACHE_EVAL)
    pred_fp32, pred_int8, true_labels = ev['pred_fp32'], ev['pred_int8'], ev['true_labels']
    acc_fp32 = (pred_fp32==true_labels).mean()
    acc_int8 = (pred_int8==true_labels).mean()
    print(f'FP32={acc_fp32:.4f} | INT8={acc_int8:.4f}')
else:
    pred_fp32, true_labels, acc_fp32 = eval_model(model_fp32, test_dl_cpu, 'FP32')
    pred_int8, _,           acc_int8 = eval_model(model_int8, test_dl_cpu, 'INT8')
    np.savez(CACHE_EVAL, pred_fp32=pred_fp32, pred_int8=pred_int8, true_labels=true_labels)
    print(f'✅ Saved -> {CACHE_EVAL.name}')

cm = confusion_matrix(true_labels, pred_int8)
fig, ax = plt.subplots(figsize=(8,7))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax)
ax.set_title('Confusion Matrix — INT8 ConvViT', fontweight='bold')
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
plt.tight_layout(); plt.savefig('confusion_matrix.png', dpi=150); plt.show()

## Phase 8 — Jamming Attack Demo (Cached)

In [ ]:
NOISE_LEVELS = [0.0, 0.05, 0.1, 0.2, 0.3, 0.5, 0.8, 1.0, 1.5, 2.0]

if CACHE_JAMMING.exists():
    print('✅ Loading jamming results from cache...')
    jm = np.load(CACHE_JAMMING)
    acc_dynamic = jm['acc_dynamic'].tolist()
    acc_static  = jm['acc_static'].tolist()
else:
    print('🔄 Running jamming simulation...')
    model_static = ConvViT(dynamic=False)
    model_static.load_state_dict(torch.load(MODEL_PATH, map_location='cpu'), strict=False)
    model_static_int8 = quantize_dynamic(model_static, {nn.Linear}, dtype=torch.qint8).eval()
    acc_dynamic, acc_static = [], []
    print(f'{"Noise":>8} | {"Dynamic":>10} | {"Static":>10}')
    for ns in NOISE_LEVELS:
        cd, cs, tot = 0, 0, 0
        with torch.no_grad():
            for x, y in test_dl_cpu:
                xn  = x + torch.randn_like(x)*ns
                cd += (model_int8(xn).argmax(1)==y).sum().item()
                cs += (model_static_int8(xn).argmax(1)==y).sum().item()
                tot+= y.size(0)
        acc_dynamic.append(cd/tot); acc_static.append(cs/tot)
        print(f'{ns:>8.2f} | {cd/tot:>10.4f} | {cs/tot:>10.4f}')
    np.savez(CACHE_JAMMING, acc_dynamic=np.array(acc_dynamic),
             acc_static=np.array(acc_static), noise_levels=np.array(NOISE_LEVELS))
    print(f'✅ Saved -> {CACHE_JAMMING.name}')

plt.figure(figsize=(10,5))
plt.plot(NOISE_LEVELS, acc_dynamic, 'o-', lw=2.5, color='royalblue',
         label='INT8 + Dynamic Scale (Ours)')
plt.plot(NOISE_LEVELS, acc_static,  's--', lw=2.5, color='tomato',
         label='INT8 Static (Baseline)')
plt.axvline(0.3, color='gray', ls=':', lw=1.5, label='Moderate Jamming')
plt.fill_between(NOISE_LEVELS, acc_dynamic, acc_static,
                 alpha=0.12, color='royalblue', label='Robustness Gap')
plt.xlabel('Jamming Noise sigma', fontsize=13)
plt.ylabel('Classification Accuracy', fontsize=13)
plt.title('Robustness Under Acoustic Jamming\nDynamic-Scale INT8 vs Static INT8',
          fontsize=12, fontweight='bold')
plt.legend(fontsize=10); plt.grid(alpha=0.3)
plt.tight_layout(); plt.savefig('jamming_robustness.png', dpi=150); plt.show()

## Phase 9 — FPGA Resource Estimation

In [ ]:
D, H_h, L = 256, 8, 3
Dh, N     = D//H_h, NUM_PATCHES+1

mac_cnn1 = 2*(3*3*1*32)  *(128*304)
mac_cnn2 = 2*(3*3*32*64) *(64*152)
mac_cnn3 = 2*(3*3*64*128)*(32*76)
mac_cnn  = mac_cnn1 + mac_cnn2 + mac_cnn3
mac_proj = NUM_PATCHES * 128 * D
mac_qkv  = L*N*D*(3*D)
mac_attn = 2*L*H_h*N*N*Dh
mac_out  = L*N*D*D
mac_mlp  = L*N*(D*4*D + 4*D*D)
mac_head = D*(D//2) + (D//2)*NUM_CLASSES
mac_tfm  = mac_qkv+mac_attn+mac_out+mac_mlp+mac_head
total_macs = mac_cnn + mac_proj + mac_tfm

cnn_bytes = sum(p.numel()*4 for p in model.cnn.parameters())
tfm_bytes = sum(p.numel()*1 for p in model.blocks.parameters())
prj_bytes = sum(p.numel()*1 for p in model.proj.parameters())
kv_bytes  = L*H_h*N*Dh*2
total_bytes = cnn_bytes+tfm_bytes+prj_bytes+kv_bytes

FPGA = {'DSP48':2520, 'BRAM_MB':32.1/8, 'freq':200e6}
dsp_cnn = min(512, int(mac_cnn/(FPGA['freq']*5)))
dsp_tfm = min(768, int(mac_tfm/(FPGA['freq']*0.5)))
dsp_tot = dsp_cnn+dsp_tfm
lat_cnn = mac_cnn/(max(dsp_cnn,1)*FPGA['freq'])*1000
lat_tfm = mac_tfm/(max(dsp_tfm,1)*FPGA['freq'])*1000
lat_tot = lat_cnn+lat_tfm

print('='*60)
print('  ConvViT FPGA Report — Zynq UltraScale+ ZU9EG @ 200MHz')
print('='*60)
print(f'  CNN MACs        : {mac_cnn/1e9:.3f} GMACs')
print(f'  Transformer MACs: {mac_tfm/1e9:.3f} GMACs')
print(f'  Total MACs      : {total_macs/1e9:.3f} GMACs')
print(f'  CNN weights     : {cnn_bytes/1e6:.2f} MB (FP32)')
print(f'  TFM weights     : {(tfm_bytes+prj_bytes)/1e6:.2f} MB (INT8)')
print(f'  KV Cache        : {kv_bytes/1e3:.1f} KB (INT8)')
print(f'  Total memory    : {total_bytes/1e6:.2f} MB / {FPGA["BRAM_MB"]:.2f} MB')
print(f'  BRAM util       : {total_bytes/1e6/FPGA["BRAM_MB"]*100:.1f}%')
print(f'  DSP (CNN+TFM)   : {dsp_tot}/{FPGA["DSP48"]} ({dsp_tot/FPGA["DSP48"]*100:.1f}%)')
print(f'  Latency         : {lat_tot:.2f} ms ({lat_cnn:.2f} CNN + {lat_tfm:.2f} TFM)')
print(f'  Throughput      : {1000/lat_tot:.0f} frames/sec')
print('='*60)

util = {'DSP48E2':dsp_tot/FPGA['DSP48']*100,
        'BRAM':total_bytes/1e6/FPGA['BRAM_MB']*100,
        'LUT (est.)':38.0}
fig, ax = plt.subplots(figsize=(9, 3.5))
bars = ax.barh(list(util.keys()), list(util.values()),
               color=['royalblue','seagreen','darkorange'], height=0.45)
ax.axvline(80,  color='red',    ls='--', lw=1.5, label='80% limit')
ax.axvline(100, color='darkred',ls='-',  lw=1.5, label='100% max')
for b, v in zip(bars, util.values()):
    ax.text(b.get_width()+1, b.get_y()+b.get_height()/2,
            f'{v:.1f}%', va='center', fontsize=11)
ax.set(xlim=(0,115), xlabel='Utilization (%)',
       title='FPGA Resource Utilization — Zynq UltraScale+ ZU9EG')
ax.legend(); ax.grid(axis='x', alpha=0.3)
plt.tight_layout(); plt.savefig('fpga_resources.png', dpi=150); plt.show()

## Phase 10 — Final Results Summary

In [ ]:
print('='*60)
print('  PASSIVE SONAR CLASSIFICATION — FINAL RESULTS')
print('='*60)
print(f'  Dataset      : ShipsEar/DS3500 ({len(all_specs)} samples, {NUM_CLASSES} classes)')
print(f'  Model        : ConvViT (CNN + DynamicScale Transformer)')
print(f'  Params       : {total:,}')
print(f'  FP32 Acc     : {acc_fp32:.4f}')
print(f'  INT8 Acc     : {acc_int8:.4f}  (drop {(acc_fp32-acc_int8)*100:.2f}%)')
print(f'  Compression  : {fp32_mb:.2f} MB -> {int8_mb:.2f} MB ({fp32_mb/int8_mb:.1f}x)')
print(f'  FPGA Latency : {lat_tot:.2f} ms/frame')
print(f'  DSP Usage    : {dsp_tot}/{FPGA["DSP48"]} ({dsp_tot/FPGA["DSP48"]*100:.1f}%)')
print('='*60)
print('\nCache files:')
for f in [CACHE_SPEC,CACHE_SPLITS,MODEL_PATH,CACHE_HISTORY,CACHE_EVAL,CACHE_JAMMING]:
    st = f'OK ({f.stat().st_size/1e6:.1f}MB)' if f.exists() else 'MISSING'
    print(f'  {str(f):40s} {st}')